# Linux & Bash basics

> **Note:** the notebooks in `examples/` are reset every time your server starts, so they are always up to date. To keep your changes, copy this notebook into `work/` (right-click it in the file browser → *Copy*, then paste it in `work/`).

This notebook uses the **Bash** kernel: every cell runs in the same Linux shell, just like typing commands in a terminal.
You'll learn how to move around the file system, work with files, combine commands, and write small scripts.

Run a cell with **Shift + Enter**.

## 1. Where am I?

- `pwd` prints the **current directory** (where you are).
- `whoami` prints your user name.
- `ls` lists the files in a directory; `ls -l` shows details, `ls -a` also shows hidden files (those starting with `.`).

In [1]:
pwd
whoami

/home/jovyan/examples


jovyan


In [2]:
ls -la ~

total 56


drwxrws--- 1 jovyan users 4096 Nov  4 08:30 .


drwxr-xr-x 1 root   root  4096 Nov  2 22:55 ..


-rw-rw-r-- 1 jovyan users  220 Feb 13  2026 .bash_logout


-rw-rw-r-- 1 jovyan users 3802 Nov  2 22:55 .bashrc


drwxrwsr-x 1 jovyan users 4096 Nov  4 08:30 .ipython


... (2 outputs left out)

drwxrwsr-x 2 jovyan users 4096 Nov  2 22:56 .mamba


-rw-rw-r-- 1 jovyan users  807 Feb 13  2026 .profile


drwxrwxrwx 2 1001   1001  4096 Nov  4 08:28 examples


drwxrwsr-x 2 jovyan users 4096 Nov  2 22:55 work


## 2. Moving around

- `cd <dir>` changes directory. `cd ~` (or just `cd`) goes home, `cd ..` goes up one level.
- Paths starting with `/` are **absolute**; others are **relative** to where you are.

We'll practise in a scratch folder under `/tmp` so we don't clutter your home.

In [3]:
mkdir -p /tmp/playground
cd /tmp/playground
pwd

/tmp/playground


## 3. Creating and viewing files

- `echo "text" > file` writes text to a file (overwrites); `>>` appends.
- `cat file` prints a file, `head -n 3` / `tail -n 3` show the first/last lines, `wc -l` counts lines.
- `touch file` creates an empty file.

In [4]:
echo "apple" > fruits.txt
echo "banana" >> fruits.txt
echo "cherry" >> fruits.txt
echo "banana" >> fruits.txt
cat fruits.txt

apple


banana


cherry


banana


In [5]:
head -n 2 fruits.txt
echo ---
wc -l fruits.txt

apple


banana


---


4 fruits.txt


## 4. Copying, moving, deleting

- `cp a b` copies, `mv a b` moves or renames, `rm file` deletes (no recycle bin!).
- `mkdir dir` creates a directory, `rm -r dir` deletes a directory and everything in it.

In [6]:
mkdir -p backup
cp fruits.txt backup/
mv backup/fruits.txt backup/fruits-copy.txt
ls backup

fruits-copy.txt


## 5. Pipes and redirection

The pipe `|` sends the output of one command into the next. This is the heart of the Linux command line:
small tools that do one thing well, combined together.

In [7]:
sort fruits.txt

apple


banana


banana


cherry


In [8]:
sort fruits.txt | uniq -c | sort -rn

      2 banana


      1 cherry


      1 apple


In [9]:
cat fruits.txt | tr 'a-z' 'A-Z' > shout.txt
cat shout.txt

APPLE


BANANA


CHERRY


BANANA


## 6. Searching: `grep` and `find`

- `grep pattern file` prints lines that match; `-i` ignores case, `-n` shows line numbers, `-c` counts.
- `find dir -name "*.txt"` searches for files by name.

In [10]:
grep -n an fruits.txt

2:banana


4:banana


In [11]:
find /tmp/playground -name "*.txt"

/tmp/playground/shout.txt


/tmp/playground/backup/fruits-copy.txt


/tmp/playground/fruits.txt


## 7. Permissions

`ls -l` shows permissions like `-rwxr-xr--`: read/write/execute for the **owner**, the **group**, and **others**.
`chmod` changes them: `chmod +x script.sh` makes a file executable, `chmod 644 file` sets `rw-r--r--`.

In [12]:
ls -l fruits.txt
chmod 600 fruits.txt
ls -l fruits.txt
chmod 644 fruits.txt

-rw-r--r-- 1 jovyan users 27 Nov  4 08:30 fruits.txt


-rw------- 1 jovyan users 27 Nov  4 08:30 fruits.txt


## 8. Variables and environment

- `name=value` creates a shell variable (**no spaces** around `=`); use it with `$name`.
- Environment variables like `$HOME`, `$USER` and `$PATH` are available to every program.
- `$(command)` inserts the output of a command.

In [13]:
course="Linux basics"
echo "Welcome to $course, $USER!"
echo "Your home is $HOME"
echo "There are $(wc -l < fruits.txt) fruits"
echo "$PATH" | tr ':' '\n' | head -n 3

Welcome to Linux basics, !


Your home is /home/jovyan


There are 4 fruits


/opt/conda/bin


/opt/conda/condabin


/opt/java/bin


## 9. Your first script

A script is a text file with commands. The first line (`#!/bin/bash`) says which program runs it.
Inside a script, `$1`, `$2`… are the arguments and `$#` is how many there are.

Below we create the script with `cat > file <<'EOF' … EOF` (a *here document*), make it executable and run it.

In [14]:
cat > greet.sh <<'EOF'
#!/bin/bash
# Usage: ./greet.sh NAME [TIMES]
name="${1:-world}"
times="${2:-1}"

if [ "$#" -eq 0 ]; then
    echo "(no name given, using the default)"
fi

for ((i = 1; i <= times; i++)); do
    echo "$i: Hello, $name!"
done
EOF
chmod +x greet.sh
./greet.sh
./greet.sh Ada 3

(no name given, using the default)


1: Hello, world!


1: Hello, Ada!


2: Hello, Ada!


3: Hello, Ada!


## 10. Conditions and loops

- `if [ condition ]; then …; elif …; else …; fi`. Common tests: `-f file` (file exists), `-d dir`, `"$a" = "$b"` (strings), `$x -lt 10` (numbers: `-eq -ne -lt -le -gt -ge`).
- `for item in list; do …; done` and `while [ condition ]; do …; done`.
- `$(( ))` does integer arithmetic.

In [15]:
for file in fruits.txt missing.txt; do
    if [ -f "$file" ]; then
        echo "$file exists with $(wc -l < "$file") lines"
    else
        echo "$file does not exist"
    fi
done

fruits.txt exists with 4 lines


missing.txt does not exist


In [16]:
count=1
while [ $count -le 5 ]; do
    echo "$count squared is $((count * count))"
    count=$((count + 1))
done

1 squared is 1


2 squared is 4


3 squared is 9


4 squared is 16


5 squared is 25


In [17]:
# Loop over the lines of a file
while read -r fruit; do
    echo "I like $fruit"
done < fruits.txt

I like apple


I like banana


I like cherry


I like banana


## Try it yourself

1. Create a file `numbers.txt` with the numbers 1 to 10, one per line (hint: `seq 1 10 > numbers.txt`).
2. Use a pipe to show only the even numbers (hint: `grep -E '[02468]$'`).
3. Write a script `count.sh` that prints how many lines the file given as its first argument has, or an error message if the file doesn't exist.

In [18]:
# Your code here
seq 1 10 > numbers.txt
cat numbers.txt | head -n 3

1


2


3


## Clean up

When you're done, remove the scratch folder:

In [19]:
cd ~
rm -rf /tmp/playground
echo "All clean"

All clean
